# Market-Implied Inflation Forecasts — Data Pipeline

**Portfolio version.** This notebook contains my end-to-end implementation of the Kalshi CPI data acquisition and forecast-construction pipeline for a UCLA MQE QuantLab research project. It retrieves market data, reconciles live and historical API schemas, standardizes contract observations, constructs fixed-horizon snapshots, and converts threshold prices into forecast distributions.


# Kalshi CPI YoY — Data & Forecast Construction Pipeline

**End-to-end construction pipeline:**
1. Pull raw events + candlesticks from Kalshi
2. Flatten into a long-format table
3. Parse real contract metadata (thresholds, contract type)
4. Build fixed-horizon snapshots (21/14/7/3/1d + final)
5. Build the forecast panel: survival curve -> monotonicity fix -> differencing -> summary stats

**Run top to bottom, in order** - each section depends on files saved by the ones before it. All output files are saved to `kalshi_data/`, same as when these were separate notebooks, so nothing about the outputs changes - this is purely a convenience merge for sharing.

---
# Data acquisition from Kalshi
---

# Kalshi CPI YoY (KXCPIYOY) Data Pull

- Live event candlesticks: one batched call per event: /series/{series}/events/{event}/candlesticks
- Historical markets list: /historical/markets?event_ticker=X
- Historical candlesticks: /historical/markets/{ticker}/candlesticks (per individual market ticker, not per event. so an old event needs one call per threshold contract instead of one batched call).

It also branches on whether an event is live or historical, and handles each path different so that everything still works. v3

In [ ]:
import requests
import time
import json
import os
import csv
from datetime import datetime, timezone

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
SERIES_TICKER = "KXCPIYOY"

OUTPUT_DIR = "kalshi_data"
CANDLES_DIR = os.path.join(OUTPUT_DIR, "candlesticks")
SUMMARY_CSV = os.path.join(OUTPUT_DIR, "data_quality_summary.csv")

REQUEST_DELAY_SECONDS = 0.3

os.makedirs(CANDLES_DIR, exist_ok=True)

## Converting ISO8601 timestamps to Unix timestamps

In [ ]:
def iso_to_unix(iso_string):
    if iso_string.endswith("Z"):
        iso_string = iso_string[:-1] + "+00:00"
    dt = datetime.fromisoformat(iso_string)
    return int(dt.timestamp())

## Step 1: Pull every event in the series (with pagination)

In [ ]:
def get_all_events(series_ticker, status="settled"):
    events = []
    cursor = ""
    while True:
        params = {"series_ticker": series_ticker, "status": status, "limit": 200}
        if cursor:
            params["cursor"] = cursor

        resp = requests.get(f"{BASE_URL}/events", params=params)
        resp.raise_for_status()
        data = resp.json()

        events.extend(data.get("events", []))
        cursor = data.get("cursor", "")
        print(f"  pulled {len(data.get('events', []))} events, running total: {len(events)}")

        if not cursor:
            break
        time.sleep(REQUEST_DELAY_SECONDS)

    return events

In [ ]:
events = get_all_events(SERIES_TICKER, status="settled")
print(f"\nTotal events found: {len(events)}")

with open(os.path.join(OUTPUT_DIR, "events.json"), "w") as f:
    json.dump(events, f, indent=2)

events[:3]

  pulled 44 events, running total: 44

Total events found: 44


[{'available_on_brokers': True,
  'category': 'Economics',
  'collateral_return_type': 'DIRECNET',
  'event_ticker': 'KXCPIYOY-26JUN',
  'exchange_index': 0,
  'last_updated_ts': '2026-05-12T14:30:46.112599Z',
  'mutually_exclusive': False,
  'series_ticker': 'KXCPIYOY',
  'settlement_sources': [{'name': 'Bureau of Labor Statistics',
    'url': 'https://www.bls.gov/cpi/'}],
  'strike_period': '',
  'sub_title': 'In Jun 2026',
  'title': 'Inflation in June 2026 (CPI YoY)'},
 {'available_on_brokers': True,
  'category': 'Economics',
  'collateral_return_type': 'DIRECNET',
  'event_ticker': 'KXCPIYOY-26MAY',
  'exchange_index': 0,
  'last_updated_ts': '2026-05-05T21:25:41.191484Z',
  'mutually_exclusive': False,
  'series_ticker': 'KXCPIYOY',
  'settlement_sources': [{'name': 'Bureau of Labor Statistics',
    'url': 'https://www.bls.gov/cpi/'}],
  'strike_period': '',
  'sub_title': 'In May 2026',
  'title': 'Inflation in May 2026 (CPI YoY)'},
 {'available_on_brokers': True,
  'category':

## Step 2: Get markets and date range (LIVE first, then HISTORICAL)

Shows which path worked, so we know how to pull candlesticks in step 3.

In [ ]:
def get_live_markets(event_ticker):
    """Try the live nested-markets path (works for events within the last ~3 months)."""
    params = {"with_nested_markets": "true"}
    resp = requests.get(f"{BASE_URL}/events/{event_ticker}", params=params)
    resp.raise_for_status()
    data = resp.json()
    event = data.get("event", data)
    return event.get("markets", [])


def get_historical_markets(event_ticker):
    """Fall back to the historical markets endpoint for older, settled events."""
    params = {"event_ticker": event_ticker, "limit": 200}
    resp = requests.get(f"{BASE_URL}/historical/markets", params=params)
    resp.raise_for_status()
    data = resp.json()
    return data.get("markets", [])


def markets_to_date_range(markets):
    open_times = [m["open_time"] for m in markets if m.get("open_time")]
    close_times = [m["close_time"] for m in markets if m.get("close_time")]
    if not open_times or not close_times:
        return None, None
    start_ts = min(iso_to_unix(t) for t in open_times)
    end_ts = max(iso_to_unix(t) for t in close_times)
    return start_ts - 86400, end_ts + 86400


def get_event_markets_and_range(event_ticker):
    """
    Try live first, then historical. Returns (markets, start_ts, end_ts, source)
    where source is 'live', 'historical', or 'none'.
    """
    markets = get_live_markets(event_ticker)
    if markets:
        start_ts, end_ts = markets_to_date_range(markets)
        if start_ts is not None:
            return markets, start_ts, end_ts, "live"

    time.sleep(REQUEST_DELAY_SECONDS)
    markets = get_historical_markets(event_ticker)
    if markets:
        start_ts, end_ts = markets_to_date_range(markets)
        if start_ts is not None:
            return markets, start_ts, end_ts, "historical"

    return [], None, None, "none"

In [ ]:
#Test on the recent event (LIVE)
test_event = events[0]["event_ticker"]
print("Testing LIVE path on:", test_event)
markets, s, e, src = get_event_markets_and_range(test_event)
print(f"source: {src}, num_markets: {len(markets)}, start_ts: {s}, end_ts: {e}")

Testing LIVE path on: KXCPIYOY-26JUN
source: live, num_markets: 21, start_ts: 1773352800, end_ts: 1784118540


In [ ]:
#Test on an old event (this should hit the HISTORICAL)
old_event = [ev["event_ticker"] for ev in events if ev["event_ticker"].startswith("CPIYOY-")][0]
print("Testing on old-format event:", old_event)
old_markets, s2, e2, src2 = get_event_markets_and_range(old_event)
print(f"source: {src2}, num_markets: {len(old_markets)}, start_ts: {s2}, end_ts: {e2}")

if src2 == "none":
    print("\nStill failing -- the historical/markets endpoint didn't return usable data either.")
    print("Worth checking the raw historical/markets response by hand at this point.")
else:
    print(f"\nWorked via '{src2}' path! Safe to run the full loop below.")
    if old_markets:
        print("Sample market ticker:", old_markets[0].get("ticker"))

Testing on old-format event: CPIYOY-24OCT
source: historical, num_markets: 9, start_ts: 1726063200, end_ts: 1731590700

Worked via 'historical' path! Safe to run the full loop below.
Sample market ticker: CPIYOY-24OCT-T3.1


## Step 3: Pull candlesticks (different call shape depending on source)

- live events: one batched call across the whole event
- historical events: one call per market ticker, then combined into the same shape

In [ ]:
def get_live_event_candlesticks(series_ticker, event_ticker, start_ts, end_ts, period_interval=1440):
    params = {"start_ts": start_ts, "end_ts": end_ts, "period_interval": period_interval}
    resp = requests.get(
        f"{BASE_URL}/series/{series_ticker}/events/{event_ticker}/candlesticks",
        params=params,
    )
    resp.raise_for_status()
    return resp.json()


def get_historical_market_candlesticks(ticker, start_ts, end_ts, period_interval=1440):
    params = {"start_ts": start_ts, "end_ts": end_ts, "period_interval": period_interval}
    resp = requests.get(f"{BASE_URL}/historical/markets/{ticker}/candlesticks", params=params)
    resp.raise_for_status()
    return resp.json()


def get_historical_event_candlesticks(markets, start_ts, end_ts, period_interval=1440):
    """
    Loop per market ticker and combine into the same {market_tickers, market_candlesticks}
    shape the live endpoint returns, so downstream code doesn't need to care which path was used.
    """
    market_tickers = []
    market_candlesticks = []
    for m in markets:
        ticker = m["ticker"]
        data = get_historical_market_candlesticks(ticker, start_ts, end_ts, period_interval)
        market_tickers.append(ticker)
        market_candlesticks.append(data.get("candlesticks", []))
        time.sleep(REQUEST_DELAY_SECONDS)
    return {"market_tickers": market_tickers, "market_candlesticks": market_candlesticks}

In [ ]:
#Test candlestick pull on the live test event
live_candles = get_live_event_candlesticks(SERIES_TICKER, test_event, s, e)
print("LIVE test --", len(live_candles["market_tickers"]), "markets")
for t, c in zip(live_candles["market_tickers"], live_candles["market_candlesticks"]):
    print(f"  {t}: {len(c)} candles")

LIVE test -- 21 markets
  KXCPIYOY-26JUN-T2.5: 84 candles
  KXCPIYOY-26JUN-T2.6: 108 candles
  KXCPIYOY-26JUN-T2.7: 100 candles
  KXCPIYOY-26JUN-T2.8: 96 candles
  KXCPIYOY-26JUN-T2.9: 99 candles
  KXCPIYOY-26JUN-T3.0: 105 candles
  KXCPIYOY-26JUN-T3.1: 105 candles
  KXCPIYOY-26JUN-T3.2: 105 candles
  KXCPIYOY-26JUN-T3.3: 110 candles
  KXCPIYOY-26JUN-T3.4: 111 candles
  KXCPIYOY-26JUN-T3.5: 108 candles
  KXCPIYOY-26JUN-T3.6: 113 candles
  KXCPIYOY-26JUN-T3.7: 117 candles
  KXCPIYOY-26JUN-T3.8: 98 candles
  KXCPIYOY-26JUN-T3.9: 95 candles
  KXCPIYOY-26JUN-T4.0: 96 candles
  KXCPIYOY-26JUN-T4.1: 61 candles
  KXCPIYOY-26JUN-T4.2: 62 candles
  KXCPIYOY-26JUN-T4.3: 58 candles
  KXCPIYOY-26JUN-T4.4: 60 candles
  KXCPIYOY-26JUN-T4.5: 57 candles


In [ ]:
#Test candlestick pull on the historical test event (only run if src2 == 'historical' above)
if src2 == "historical":
    hist_candles = get_historical_event_candlesticks(old_markets, s2, e2)
    print("HISTORICAL test --", len(hist_candles["market_tickers"]), "markets")
    for t, c in zip(hist_candles["market_tickers"], hist_candles["market_candlesticks"]):
        print(f"  {t}: {len(c)} candles")
else:
    print("Skipping -- old event didn't resolve via the historical path, see previous cell.")

HISTORICAL test -- 9 markets
  CPIYOY-24OCT-T3.1: 15 candles
  CPIYOY-24OCT-T3.0: 18 candles
  CPIYOY-24OCT-T2.9: 20 candles
  CPIYOY-24OCT-T2.8: 16 candles
  CPIYOY-24OCT-T2.7: 34 candles
  CPIYOY-24OCT-T2.6: 37 candles
  CPIYOY-24OCT-T2.5: 33 candles
  CPIYOY-24OCT-T2.4: 37 candles
  CPIYOY-24OCT-T2.3: 30 candles


## Step 4: Run the full pull across all events

Only run this once both test cells above show real data (not none, not 0 candles).

In [ ]:
def summarize_event_pull(event_ticker, candle_data, date_source):
    tickers = candle_data.get("market_tickers", [])
    candle_lists = candle_data.get("market_candlesticks", [])
    counts = [len(c) for c in candle_lists]
    empty_count = sum(1 for c in counts if c == 0)
    return {
        "event_ticker": event_ticker,
        "date_source": date_source,
        "num_markets": len(tickers),
        "min_candles": min(counts) if counts else 0,
        "max_candles": max(counts) if counts else 0,
        "avg_candles": round(sum(counts) / len(counts), 1) if counts else 0,
        "empty_markets": empty_count,
    }

In [ ]:
summary_rows = []

for i, event in enumerate(events, start=1):
    event_ticker = event["event_ticker"]
    print(f"[{i}/{len(events)}] {event_ticker}")

    try:
        markets, s_ts, e_ts, date_source = get_event_markets_and_range(event_ticker)
        time.sleep(REQUEST_DELAY_SECONDS)

        if date_source == "none":
            print("    -> no market data found via live OR historical, skipping")
            summary_rows.append({
                "event_ticker": event_ticker, "date_source": "none", "num_markets": 0,
                "min_candles": 0, "max_candles": 0, "avg_candles": 0,
                "empty_markets": 0, "error": "no_date_range_found",
            })
            continue

        if date_source == "live":
            candles = get_live_event_candlesticks(SERIES_TICKER, event_ticker, s_ts, e_ts)
        else:  # historical
            candles = get_historical_event_candlesticks(markets, s_ts, e_ts)

        out_path = os.path.join(CANDLES_DIR, f"{event_ticker}.json")
        with open(out_path, "w") as f:
            json.dump(candles, f, indent=2)

        row = summarize_event_pull(event_ticker, candles, date_source)
        row["error"] = ""
        summary_rows.append(row)
        print(f"    -> [{date_source}] {row['num_markets']} markets, avg {row['avg_candles']} candles, {row['empty_markets']} empty")

    except requests.exceptions.HTTPError as e:
        print(f"    -> HTTP error: {e}")
        summary_rows.append({
            "event_ticker": event_ticker, "date_source": "error", "num_markets": 0,
            "min_candles": 0, "max_candles": 0, "avg_candles": 0,
            "empty_markets": 0, "error": str(e),
        })

print("\nDone pulling all events.")

[1/44] KXCPIYOY-26JUN
    -> [live] 21 markets, avg 92.8 candles, 0 empty
[2/44] KXCPIYOY-26MAY
    -> [live] 27 markets, avg 69.0 candles, 0 empty
[3/44] KXCPIYOY-26APR
    -> [historical] 23 markets, avg 54.5 candles, 0 empty
[4/44] KXCPIYOY-26MAR
    -> [historical] 23 markets, avg 40.9 candles, 0 empty
[5/44] KXCPIYOY-26FEB
    -> [historical] 10 markets, avg 49.5 candles, 0 empty
[6/44] KXCPIYOY-26JAN
    -> [historical] 11 markets, avg 31.7 candles, 0 empty
[7/44] KXCPIYOY-25DEC
    -> [historical] 8 markets, avg 41.5 candles, 0 empty
[8/44] KXCPIYOY-25NOV
    -> [historical] 8 markets, avg 53.6 candles, 0 empty
[9/44] KXCPIYOY-25OCT
    -> [historical] 8 markets, avg 54.2 candles, 0 empty
[10/44] KXCPIYOY-25SEP
    -> [historical] 8 markets, avg 60.4 candles, 0 empty
[11/44] KXCPIYOY-25AUG
    -> [historical] 8 markets, avg 54.8 candles, 0 empty
[12/44] KXCPIYOY-25JUL
    -> [historical] 8 markets, avg 59.4 candles, 0 empty
[13/44] KXCPIYOY-25JUN
    -> [historical] 11 markets, 

## Step 5: Write the summary CSV

In [ ]:
fieldnames = ["event_ticker", "date_source", "num_markets", "min_candles", "max_candles",
              "avg_candles", "empty_markets", "error"]

with open(SUMMARY_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in summary_rows:
        row.setdefault("error", "")
        writer.writerow(row)

print(f"Summary written to {SUMMARY_CSV}")
print(f"Raw candlestick JSON files saved under {CANDLES_DIR}/")

Summary written to kalshi_data/data_quality_summary.csv
Raw candlestick JSON files saved under kalshi_data/candlesticks/


In [ ]:
import pandas as pd
summary_df = pd.DataFrame(summary_rows)
print("Rows by date_source:")
print(summary_df["date_source"].value_counts())
print()
summary_df

Rows by date_source:
date_source
historical    42
live           2
Name: count, dtype: int64



,event_ticker,date_source,num_markets,min_candles,max_candles,avg_candles,empty_markets,error
0,KXCPIYOY-26JUN,live,21,57,117,92.8,0,
1,KXCPIYOY-26MAY,live,27,34,92,69.0,0,
2,KXCPIYOY-26APR,historical,23,29,68,54.5,0,
3,KXCPIYOY-26MAR,historical,23,28,53,40.9,0,
4,KXCPIYOY-26FEB,historical,10,27,58,49.5,0,
5,KXCPIYOY-26JAN,historical,11,31,32,31.7,0,
6,KXCPIYOY-25DEC,historical,8,40,42,41.5,0,
7,KXCPIYOY-25NOV,historical,8,48,61,53.6,0,
8,KXCPIYOY-25OCT,historical,8,47,61,54.2,0,
9,KXCPIYOY-25SEP,historical,8,53,64,60.4,0,


---
# Forecast construction, Step 1 - Flatten candlesticks
---

# Kalshi CPI YoY - Parse & Flatten Candlesticks (v2)

**What changed from v1:** the first version silently returned 100% missing bid/ask/price for every historical event (42 of 44). The cause: live candlesticks use field names like `close_dollars`, but historical candlesticks use `close` (no suffix) - two different schemas for the same kind of data. v1 only checked for `close_dollars`, so it found nothing on every historical row even though the data was there the whole time. This version checks both naming conventions.

**What each output row contains:**
- `event_ticker` / `market_ticker` - which release and which threshold contract
- `threshold` - the CPI% value parsed from the ticker (e.g. 3.7)
- `date` - the day this row covers
- `yes_bid_close` / `yes_ask_close` - the closing bid/ask quotes that day
- `price_close` - the last-traded price that day (blank if no trade happened)
- `mid_quote` - (bid + ask) / 2 - our probability proxy for days with no trade
- `spread` - ask minus bid - wide spread means a less trustworthy quote
- `volume` - how many contracts actually traded that day

In [ ]:
import json
import os
import re
import csv
from datetime import datetime, timezone

CANDLES_DIR = os.path.join("kalshi_data", "candlesticks")
OUTPUT_CSV = os.path.join("kalshi_data", "flattened_candlesticks.csv")

print("Looking in:", os.path.abspath(CANDLES_DIR))
files = [f for f in os.listdir(CANDLES_DIR) if f.endswith(".json")]
print(f"Found {len(files)} event files")

## Parse threshold from ticker, and read price fields under either schema

In [ ]:
def parse_threshold(market_ticker):
    """Extract the threshold value from a ticker like 'KXCPIYOY-26JUN-T3.5' -> 3.5"""
    match = re.search(r"-T(-?\d+\.?\d*)$", market_ticker)
    return float(match.group(1)) if match else None


def get_field(obj, base_name):
    """
    Live candlesticks name fields like 'close_dollars'.
    Historical candlesticks name the SAME field just 'close' (no suffix).
    Try both so this works regardless of which endpoint the data came from.
    """
    if obj is None:
        return None
    if f"{base_name}_dollars" in obj:
        return obj[f"{base_name}_dollars"]
    if base_name in obj:
        return obj[base_name]
    return None


# sanity checks
print(parse_threshold("KXCPIYOY-26JUN-T3.5"))                       # expect 3.5
print(get_field({"close_dollars": "0.56"}, "close"))                 # live schema  -> 0.56
print(get_field({"close": "0.01"}, "close"))                        # historical schema -> 0.01

3.5
0.56
0.01


## Flatten one event file into a list of rows

In [ ]:
def flatten_event_file(filepath, event_ticker):
    with open(filepath) as f:
        data = json.load(f)

    rows = []
    tickers = data.get("market_tickers", [])
    candle_lists = data.get("market_candlesticks", [])

    for market_ticker, candles in zip(tickers, candle_lists):
        threshold = parse_threshold(market_ticker)

        for c in candles:
            ts = c.get("end_period_ts")
            date_str = (
                datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d")
                if ts else None
            )

            yes_bid_close = get_field(c.get("yes_bid"), "close")
            yes_ask_close = get_field(c.get("yes_ask"), "close")
            price_close = get_field(c.get("price"), "close")
            volume = c.get("volume_fp", c.get("volume"))

            mid_quote = None
            spread = None
            if yes_bid_close is not None and yes_ask_close is not None:
                bid = float(yes_bid_close)
                ask = float(yes_ask_close)
                mid_quote = round((bid + ask) / 2, 4)
                spread = round(ask - bid, 4)

            rows.append({
                "event_ticker": event_ticker,
                "market_ticker": market_ticker,
                "threshold": threshold,
                "date": date_str,
                "end_period_ts": ts,
                "yes_bid_close": yes_bid_close,
                "yes_ask_close": yes_ask_close,
                "price_close": price_close,
                "mid_quote": mid_quote,
                "spread": spread,
                "volume": volume,
            })

    return rows

In [ ]:
# Test on a LIVE event and a HISTORICAL event before running all 44
live_test = [f for f in files if f.startswith("KXCPIYOY-26JUN") or f.startswith("KXCPIYOY-26MAY")][0]
hist_test = [f for f in files if f.startswith("CPIYOY-") or (f.startswith("KXCPIYOY-") and "26JUN" not in f and "26MAY" not in f)][0]

for fname in [live_test, hist_test]:
    event_ticker = fname.replace(".json", "")
    rows = flatten_event_file(os.path.join(CANDLES_DIR, fname), event_ticker)
    missing = sum(1 for r in rows if r["mid_quote"] is None)
    print(f"{event_ticker}: {len(rows)} rows, {missing} missing mid_quote ({100*missing/len(rows):.1f}%)")

print("\nBoth should now show a reasonable, non-zero mid_quote rate (not 100% missing).")

KXCPIYOY-26JUN: 1948 rows, 0 missing mid_quote (0.0%)
CPIYOY-23DEC: 117 rows, 0 missing mid_quote (0.0%)

Both should now show a reasonable, non-zero mid_quote rate (not 100% missing).


## Run it across all 44 files and save one combined csv

In [ ]:
all_rows = []

for fname in sorted(files):
    event_ticker = fname.replace(".json", "")
    filepath = os.path.join(CANDLES_DIR, fname)
    rows = flatten_event_file(filepath, event_ticker)
    all_rows.extend(rows)
    missing = sum(1 for r in rows if r["mid_quote"] is None)
    print(f"{event_ticker}: {len(rows)} rows, {missing} missing mid_quote ({100*missing/len(rows) if rows else 0:.1f}%)")

print(f"\nTotal rows across all events: {len(all_rows)}")

CPIYOY-22DEC: 249 rows, 0 missing mid_quote (0.0%)
CPIYOY-22NOV: 68 rows, 0 missing mid_quote (0.0%)
CPIYOY-23APR: 344 rows, 0 missing mid_quote (0.0%)
CPIYOY-23AUG: 454 rows, 0 missing mid_quote (0.0%)
CPIYOY-23DEC: 117 rows, 0 missing mid_quote (0.0%)
CPIYOY-23FEB: 261 rows, 0 missing mid_quote (0.0%)
CPIYOY-23JAN: 159 rows, 0 missing mid_quote (0.0%)
CPIYOY-23JUL: 591 rows, 0 missing mid_quote (0.0%)
CPIYOY-23JUN: 455 rows, 0 missing mid_quote (0.0%)
CPIYOY-23MAR: 472 rows, 0 missing mid_quote (0.0%)
CPIYOY-23MAY: 387 rows, 0 missing mid_quote (0.0%)
CPIYOY-23NOV: 432 rows, 0 missing mid_quote (0.0%)
CPIYOY-23OCT: 385 rows, 0 missing mid_quote (0.0%)
CPIYOY-23SEP: 441 rows, 0 missing mid_quote (0.0%)
CPIYOY-24APR: 327 rows, 0 missing mid_quote (0.0%)
CPIYOY-24AUG: 208 rows, 0 missing mid_quote (0.0%)
CPIYOY-24FEB: 290 rows, 0 missing mid_quote (0.0%)
CPIYOY-24JAN: 180 rows, 0 missing mid_quote (0.0%)
CPIYOY-24JUL: 232 rows, 0 missing mid_quote (0.0%)
CPIYOY-24JUN: 282 rows, 0 missin

In [ ]:
fieldnames = ["event_ticker", "market_ticker", "threshold", "date", "end_period_ts",
              "yes_bid_close", "yes_ask_close", "price_close", "mid_quote", "spread", "volume"]

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

print(f"Saved {len(all_rows)} rows to {OUTPUT_CSV}")

Saved 20137 rows to kalshi_data/flattened_candlesticks.csv


## Checks to confirm the fix actually worked across the full dataset

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
print("Shape:", df.shape)
print("Events covered:", df["event_ticker"].nunique(), "(should be 44)")
print("Date range:", df["date"].min(), "to", df["date"].max())
print()
print("Overall missing mid_quote rate:")
print(f"  {df['mid_quote'].isna().sum()} of {len(df)} ({100*df['mid_quote'].isna().mean():.1f}%)")
print()
print("Missing rate BY EVENT -- should no longer be a clean 0%/100% split:")
by_event = df.groupby("event_ticker").agg(
    rows=("mid_quote", "size"),
    missing=("mid_quote", lambda x: x.isna().sum())
)
by_event["pct_missing"] = 100 * by_event["missing"] / by_event["rows"]
by_event.sort_values("pct_missing", ascending=False)

Shape: (20137, 11)
Events covered: 44 (should be 44)
Date range: 2022-12-08 to 2026-07-15

Overall missing mid_quote rate:
  0 of 20137 (0.0%)

Missing rate BY EVENT -- should no longer be a clean 0%/100% split:


,rows,missing,pct_missing
event_ticker,,,
CPIYOY-22DEC,249,0,0.0
CPIYOY-22NOV,68,0,0.0
KXCPIYOY-24DEC,303,0,0.0
KXCPIYOY-24NOV,276,0,0.0
KXCPIYOY-25APR,533,0,0.0
KXCPIYOY-25AUG,438,0,0.0
KXCPIYOY-25DEC,332,0,0.0
KXCPIYOY-25FEB,280,0,0.0
KXCPIYOY-25JAN,461,0,0.0


---
# Forecast construction, Step 2 - Contract parser / Market metadata
---

# Kalshi CPI YoY - Contract Parser (market_metadata)

**Also handled here:** distinguishing 'Above' (threshold ladder) contracts from 'Exactly' (mutually exclusive bin) contracts. We get this for free from the event-level `mutually_exclusive` flag already sitting in `events.json` - no extra parsing needed.

**Output:** `market_metadata.csv` - one row per contract, with its real threshold, contract type, and the rules text, ready to join against the flattened candlestick data.

In [ ]:
import requests
import time
import json
import os
import re
import csv

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
OUTPUT_DIR = "kalshi_data"
METADATA_CSV = os.path.join(OUTPUT_DIR, "market_metadata.csv")
REQUEST_DELAY_SECONDS = 0.3

with open(os.path.join(OUTPUT_DIR, "events.json")) as f:
    events = json.load(f)

print(f"Loaded {len(events)} events from events.json")

# mutually_exclusive is already here -- no extra API call needed for contract type
mutex_counts = {}
for e in events:
    mutex_counts[e["mutually_exclusive"]] = mutex_counts.get(e["mutually_exclusive"], 0) + 1
print("mutually_exclusive breakdown:", mutex_counts)

Loaded 44 events from events.json
mutually_exclusive breakdown: {False: 44}


## Fetch each event's real market list (reusing the live/historical fallback from the pull notebook)

This gets us the actual contract metadata - `floor_strike`, `cap_strike`, `rules_primary`, etc - not just the ticker string.

In [ ]:
def get_live_markets(event_ticker):
    params = {"with_nested_markets": "true"}
    resp = requests.get(f"{BASE_URL}/events/{event_ticker}", params=params)
    resp.raise_for_status()
    data = resp.json()
    event = data.get("event", data)
    return event.get("markets", [])


def get_historical_markets(event_ticker):
    params = {"event_ticker": event_ticker, "limit": 200}
    resp = requests.get(f"{BASE_URL}/historical/markets", params=params)
    resp.raise_for_status()
    data = resp.json()
    return data.get("markets", [])


def get_markets_for_event(event_ticker):
    markets = get_live_markets(event_ticker)
    if markets:
        return markets, "live"
    time.sleep(REQUEST_DELAY_SECONDS)
    markets = get_historical_markets(event_ticker)
    return markets, "historical" if markets else "none"

In [ ]:
# TEST FIRST: look at one real market object to confirm floor_strike/cap_strike are actually populated
# (the docs schema page only shows placeholder values, so we need to confirm on real data)
test_markets, test_source = get_markets_for_event(events[0]["event_ticker"])
print("source:", test_source)
print(json.dumps(test_markets[0], indent=2)[:1200])

source: live
{
  "can_close_early": true,
  "close_time": "2026-07-14T12:29:00Z",
  "created_time": "2026-03-13T20:34:45.08591Z",
  "event_ticker": "KXCPIYOY-26JUN",
  "exchange_index": 0,
  "expected_expiration_time": "2026-07-14T14:00:00Z",
  "expiration_time": "2026-10-13T14:00:00Z",
  "expiration_value": "3.5%",
  "floor_strike": 2.5,
  "last_price_dollars": "0.9900",
  "latest_expiration_time": "2026-10-13T14:00:00Z",
  "liquidity_dollars": "0.0000",
  "market_type": "binary",
  "no_ask_dollars": "1.0000",
  "no_bid_dollars": "0.0000",
  "no_sub_title": "Above 2.5%",
  "notional_value_dollars": "1.0000",
  "occurrence_datetime": "2026-07-14T12:28:57.621Z",
  "open_interest_fp": "9118.75",
  "open_time": "2026-03-13T22:00:00Z",
  "previous_price_dollars": "0.9900",
  "previous_yes_ask_dollars": "1.0000",
  "previous_yes_bid_dollars": "0.0000",
  "price_level_structure": "linear_cent",
  "price_ranges": [
    {
      "end": "1.0000",
      "start": "0.0000",
      "step": "0.0100"
 

## Parse threshold with a real hard-fail

Primary source: `floor_strike` (or `cap_strike` if floor is null - some contracts are capped from above instead of floored from below). Fallback: parse from ticker. If neither works, **raise an error** instead of silently writing a bad row - per the plan doc's 'hard-fail on unexpected wording or structural changes.'

In [ ]:
class ThresholdParseError(Exception):
    pass


def parse_threshold_from_ticker(market_ticker):
    match = re.search(r"-T(-?\d+\.?\d*)$", market_ticker)
    return float(match.group(1)) if match else None


def get_threshold(market):
    """
    Try floor_strike, then cap_strike (real metadata), then fall back to the
    ticker string. Hard-fail if none of these produce a value.
    """
    if market.get("floor_strike") is not None:
        return market["floor_strike"], "floor_strike"
    if market.get("cap_strike") is not None:
        return market["cap_strike"], "cap_strike"

    ticker_value = parse_threshold_from_ticker(market["ticker"])
    if ticker_value is not None:
        return ticker_value, "ticker_fallback"

    raise ThresholdParseError(
        f"Could not determine threshold for {market['ticker']} -- "
        f"no floor_strike, cap_strike, or parseable ticker suffix. "
        f"Stopping rather than guessing."
    )

## Build market_metadata.csv across all 44 events

In [ ]:
event_mutex_lookup = {e["event_ticker"]: e["mutually_exclusive"] for e in events}

metadata_rows = []

for i, event in enumerate(events, start=1):
    event_ticker = event["event_ticker"]
    print(f"[{i}/{len(events)}] {event_ticker}")

    markets, source = get_markets_for_event(event_ticker)
    time.sleep(REQUEST_DELAY_SECONDS)

    if not markets:
        print(f"    -> WARNING: no markets found at all for {event_ticker}, skipping")
        continue

    contract_type = "exactly" if event_mutex_lookup[event_ticker] else "above"

    for m in markets:
        threshold, threshold_source = get_threshold(m)  # raises ThresholdParseError if it can't

        metadata_rows.append({
            "event_ticker": event_ticker,
            "market_ticker": m["ticker"],
            "contract_type": contract_type,
            "threshold": threshold,
            "threshold_source": threshold_source,
            "open_time": m.get("open_time"),
            "close_time": m.get("close_time"),
            "rules_primary": m.get("rules_primary"),
            "market_source": source,
        })

print(f"\nDone. {len(metadata_rows)} market metadata rows built.")

[1/44] KXCPIYOY-26JUN
[2/44] KXCPIYOY-26MAY
[3/44] KXCPIYOY-26APR
[4/44] KXCPIYOY-26MAR
[5/44] KXCPIYOY-26FEB
[6/44] KXCPIYOY-26JAN
[7/44] KXCPIYOY-25DEC
[8/44] KXCPIYOY-25NOV
[9/44] KXCPIYOY-25OCT
[10/44] KXCPIYOY-25SEP
[11/44] KXCPIYOY-25AUG
[12/44] KXCPIYOY-25JUL
[13/44] KXCPIYOY-25JUN
[14/44] KXCPIYOY-25MAY
[15/44] KXCPIYOY-25APR
[16/44] KXCPIYOY-25MAR
[17/44] KXCPIYOY-25FEB
[18/44] KXCPIYOY-25JAN
[19/44] KXCPIYOY-24DEC
[20/44] KXCPIYOY-24NOV
[21/44] CPIYOY-24OCT
[22/44] CPIYOY-24SEP
[23/44] CPIYOY-24AUG
[24/44] CPIYOY-24JUL
[25/44] CPIYOY-24JUN
[26/44] CPIYOY-24MAY
[27/44] CPIYOY-24APR
[28/44] CPIYOY-24MAR
[29/44] CPIYOY-24FEB
[30/44] CPIYOY-24JAN
[31/44] CPIYOY-23DEC
[32/44] CPIYOY-23NOV
[33/44] CPIYOY-23OCT
[34/44] CPIYOY-23SEP
[35/44] CPIYOY-23AUG
[36/44] CPIYOY-23JUL
[37/44] CPIYOY-23JUN
[38/44] CPIYOY-23MAY
[39/44] CPIYOY-23APR
[40/44] CPIYOY-23MAR
[41/44] CPIYOY-23FEB
[42/44] CPIYOY-23JAN
[43/44] CPIYOY-22DEC
[44/44] CPIYOY-22NOV

Done. 603 market metadata rows built.


In [ ]:
fieldnames = ["event_ticker", "market_ticker", "contract_type", "threshold",
              "threshold_source", "open_time", "close_time", "rules_primary", "market_source"]

with open(METADATA_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(metadata_rows)

print(f"Saved to {METADATA_CSV}")

Saved to kalshi_data/market_metadata.csv


## Check how many thresholds actually came from real metadata vs. the ticker fallback?

In [ ]:
import pandas as pd

meta_df = pd.read_csv(METADATA_CSV)
print("Total rows:", len(meta_df))
print()
print("threshold_source breakdown (want mostly floor_strike/cap_strike, not ticker_fallback):")
print(meta_df["threshold_source"].value_counts())
print()
print("contract_type breakdown:")
print(meta_df["contract_type"].value_counts())
print()
meta_df.head(10)

Total rows: 603

threshold_source breakdown (want mostly floor_strike/cap_strike, not ticker_fallback):
threshold_source
floor_strike       545
ticker_fallback     58
Name: count, dtype: int64

contract_type breakdown:
contract_type
above    603
Name: count, dtype: int64



,event_ticker,market_ticker,contract_type,threshold,threshold_source,open_time,close_time,rules_primary,market_source
0,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.5,above,2.5,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
1,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.6,above,2.6,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
2,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.7,above,2.7,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
3,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.8,above,2.8,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
4,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.9,above,2.9,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
5,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T3.0,above,3.0,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
6,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T3.1,above,3.1,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
7,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T3.2,above,3.2,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
8,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T3.3,above,3.3,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live
9,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T3.4,above,3.4,floor_strike,2026-03-13T22:00:00Z,2026-07-14T12:29:00Z,If the Consumer Price Index (CPI) increases by...,live


---
# Forecast construction, Step 3 - Fixed-horizon snapshots
---

# Kalshi CPI YoY - Fixed-Horizon Snapshots

**What this does:** for every contract in every event, pulls out just the price observation closest to (but not after) each of the standardized horizons: 21, 14, 7, 3, and 1 calendar days before the release, plus the final eligible pre-release observation.

**Inputs:**
- `market_metadata.csv` - gives us each contract's real `close_time` (the release date)
- `flattened_candlesticks.csv` - gives us the daily mid-quote history to snapshot from

**Output:** `fixed_horizon_snapshots.csv` - one row per (contract, horizon), including an `is_stale` flag for snapshots built from a quote that's more than 14 days older than its target horizon - some early, illiquid contracts have real multi-week/month gaps with no recorded candle at all, and this flag travels with the data through every downstream step instead of relying on someone remembering to check `days_stale` by hand.

In [ ]:
import pandas as pd
from datetime import timedelta
import os

OUTPUT_DIR = "kalshi_data"
META_CSV = os.path.join(OUTPUT_DIR, "market_metadata.csv")
FLAT_CSV = os.path.join(OUTPUT_DIR, "flattened_candlesticks.csv")
SNAPSHOT_CSV = os.path.join(OUTPUT_DIR, "fixed_horizon_snapshots.csv")

HORIZONS = [21, 14, 7, 3, 1]
STALE_THRESHOLD_DAYS = 14  # anything older than this is flagged as unreliable

meta = pd.read_csv(META_CSV)
flat = pd.read_csv(FLAT_CSV)

meta["release_date"] = pd.to_datetime(meta["close_time"]).dt.date
flat["date"] = pd.to_datetime(flat["date"]).dt.date

print(f"Loaded {len(meta)} contracts from metadata, {len(flat)} price rows from flattened data")

Loaded 603 contracts from metadata, 20137 price rows from flattened data


## Core snapshotting function - test it on one real contract first

In [ ]:
def build_snapshots_for_market(market_ticker, release_date, market_flat_rows):
    """
    market_flat_rows: the flattened price rows for just this one contract, any order.
    Returns one dict per horizon (21d, 14d, 7d, 3d, 1d, final).
    """
    rows = market_flat_rows.sort_values("date")
    snapshots = []

    for h in HORIZONS:
        target_date = release_date - timedelta(days=h)
        eligible = rows[rows["date"] <= target_date]

        if eligible.empty:
            snapshots.append({
                "horizon": f"{h}d", "target_date": target_date, "actual_date": None,
                "days_stale": None, "mid_quote": None, "spread": None, "volume": None,
                "is_missing": True,
            })
        else:
            r = eligible.iloc[-1]  # most recent eligible row before/at the cutoff
            days_stale = (target_date - r["date"]).days
            snapshots.append({
                "horizon": f"{h}d", "target_date": target_date, "actual_date": r["date"],
                "days_stale": days_stale, "mid_quote": r["mid_quote"], "spread": r["spread"],
                "volume": r["volume"], "is_missing": False,
            })

    # 'final' = last available observation strictly before the release itself
    final_eligible = rows[rows["date"] < release_date]
    if final_eligible.empty:
        snapshots.append({
            "horizon": "final", "target_date": release_date, "actual_date": None,
            "days_stale": None, "mid_quote": None, "spread": None, "volume": None,
            "is_missing": True,
        })
    else:
        r = final_eligible.iloc[-1]
        days_stale = (release_date - r["date"]).days
        snapshots.append({
            "horizon": "final", "target_date": release_date, "actual_date": r["date"],
            "days_stale": days_stale, "mid_quote": r["mid_quote"], "spread": r["spread"],
            "volume": r["volume"], "is_missing": False,
        })

    return snapshots

In [ ]:
# Test on one real contract before running the full loop
test_row = meta.iloc[0]
test_ticker = test_row["market_ticker"]
test_release = test_row["release_date"]
market_flat = flat[flat["market_ticker"] == test_ticker]

print(f"Testing on {test_ticker}, release_date={test_release}, {len(market_flat)} price rows available")
for s in build_snapshots_for_market(test_ticker, test_release, market_flat):
    print(s)

Testing on KXCPIYOY-26JUN-T2.5, release_date=2026-07-14, 84 price rows available
{'horizon': '21d', 'target_date': datetime.date(2026, 6, 23), 'actual_date': datetime.date(2026, 6, 21), 'days_stale': 2, 'mid_quote': np.float64(0.985), 'spread': np.float64(0.01), 'volume': np.float64(24.75), 'is_missing': False}
{'horizon': '14d', 'target_date': datetime.date(2026, 6, 30), 'actual_date': datetime.date(2026, 6, 24), 'days_stale': 6, 'mid_quote': np.float64(0.995), 'spread': np.float64(0.01), 'volume': np.float64(1.0), 'is_missing': False}
{'horizon': '7d', 'target_date': datetime.date(2026, 7, 7), 'actual_date': datetime.date(2026, 7, 3), 'days_stale': 4, 'mid_quote': np.float64(0.995), 'spread': np.float64(0.01), 'volume': np.float64(4879.0), 'is_missing': False}
{'horizon': '3d', 'target_date': datetime.date(2026, 7, 11), 'actual_date': datetime.date(2026, 7, 11), 'days_stale': 0, 'mid_quote': np.float64(0.995), 'spread': np.float64(0.01), 'volume': np.float64(0.0), 'is_missing': False

## Run it across every contract in market_metadata.csv

In [ ]:
all_snapshot_rows = []

for i, row in meta.iterrows():
    market_ticker = row["market_ticker"]
    event_ticker = row["event_ticker"]
    release_date = row["release_date"]
    threshold = row["threshold"]

    market_flat = flat[flat["market_ticker"] == market_ticker]

    if market_flat.empty:
        print(f"WARNING: no flattened price data found for {market_ticker}, skipping")
        continue

    snaps = build_snapshots_for_market(market_ticker, release_date, market_flat)
    for s in snaps:
        s["event_ticker"] = event_ticker
        s["market_ticker"] = market_ticker
        s["threshold"] = threshold
        all_snapshot_rows.append(s)

    if (i + 1) % 100 == 0:
        print(f"  processed {i + 1}/{len(meta)} contracts...")

print(f"\nDone. Built {len(all_snapshot_rows)} snapshot rows "
      f"({len(meta)} contracts x 6 horizons = {len(meta) * 6} expected)")

  processed 100/603 contracts...
  processed 200/603 contracts...
  processed 300/603 contracts...
  processed 400/603 contracts...
  processed 500/603 contracts...
  processed 600/603 contracts...

Done. Built 3618 snapshot rows (603 contracts x 6 horizons = 3618 expected)


In [ ]:
snap_df = pd.DataFrame(all_snapshot_rows)

cols = ["event_ticker", "market_ticker", "threshold", "horizon", "target_date",
        "actual_date", "days_stale", "mid_quote", "spread", "volume", "is_missing"]
snap_df = snap_df[cols]

print(f"Built {len(snap_df)} rows")

Built 3618 rows


## Flag unreliable (very stale) snapshots

In [ ]:
snap_df["is_stale"] = snap_df["days_stale"] > STALE_THRESHOLD_DAYS

print(f"Flagged {snap_df['is_stale'].sum()} of {len(snap_df)} rows as stale (> {STALE_THRESHOLD_DAYS} days)")
print()
print("Stale rows by horizon:")
print(snap_df.groupby("horizon")["is_stale"].sum())
print()
print("Events with the most stale-flagged rows (worth double-checking these specifically):")
print(snap_df[snap_df["is_stale"]]["event_ticker"].value_counts().head(10))

Flagged 321 of 3618 rows as stale (> 14 days)

Stale rows by horizon:
horizon
14d      26
1d       66
21d      39
3d       68
7d       55
final    67
Name: is_stale, dtype: int64

Events with the most stale-flagged rows (worth double-checking these specifically):
event_ticker
CPIYOY-23JUL    86
CPIYOY-23MAR    56
CPIYOY-23JUN    35
CPIYOY-23AUG    26
CPIYOY-23APR    25
CPIYOY-23NOV    21
CPIYOY-23MAY    21
CPIYOY-23JAN    14
CPIYOY-24OCT    11
CPIYOY-23OCT    10
Name: count, dtype: int64


In [ ]:
snap_df.to_csv(SNAPSHOT_CSV, index=False)
print(f"Saved to {SNAPSHOT_CSV}")

Saved to kalshi_data/fixed_horizon_snapshots.csv


## Checks how much data is actually missing at each horizon

In [ ]:
print("Missing rate by horizon (expect MORE missing at 21d than at 1d/final --\n"
      "older/shorter-lived markets won't have data 21 days out):")
missing_by_horizon = snap_df.groupby("horizon")["is_missing"].mean() * 100
print(missing_by_horizon.reindex(["21d", "14d", "7d", "3d", "1d", "final"]))
print()

print("days_stale distribution for non-missing rows:")
print(snap_df[~snap_df["is_missing"]]["days_stale"].describe())
print()

print("mid_quote sanity check -- low thresholds should skew near 1.0, high thresholds near 0.0:")
print(snap_df.groupby(pd.cut(snap_df["threshold"], bins=5))["mid_quote"].mean())

Missing rate by horizon (expect MORE missing at 21d than at 1d/final --
older/shorter-lived markets won't have data 21 days out):
horizon
21d      3.316750
14d      2.155887
7d       2.155887
3d       0.000000
1d       0.000000
final    0.000000
Name: is_missing, dtype: float64

days_stale distribution for non-missing rows:
count    3572.000000
mean        3.612822
std         7.879368
min         0.000000
25%         0.000000
50%         0.000000
75%         3.000000
max        98.000000
Name: days_stale, dtype: float64

mid_quote sanity check -- low thresholds should skew near 1.0, high thresholds near 0.0:
threshold
(1.494, 2.8]    0.745851
(2.8, 4.1]      0.408616
(4.1, 5.4]      0.384214
(5.4, 6.7]      0.441420
(6.7, 8.0]      0.183218
Name: mid_quote, dtype: float64


/var/folders/vb/m7hw60gj479gs79ks94n24nr0000gn/T/ipykernel_39542/1655276963.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(snap_df.groupby(pd.cut(snap_df["threshold"], bins=5))["mid_quote"].mean())


In [ ]:
snap_df.head(12)

,event_ticker,market_ticker,threshold,horizon,target_date,actual_date,days_stale,mid_quote,spread,volume,is_missing,is_stale
0,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.5,2.5,21d,2026-06-23,2026-06-21,2.0,0.985,0.01,24.75,False,False
1,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.5,2.5,14d,2026-06-30,2026-06-24,6.0,0.995,0.01,1.00,False,False
2,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.5,2.5,7d,2026-07-07,2026-07-03,4.0,0.995,0.01,4879.00,False,False
3,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.5,2.5,3d,2026-07-11,2026-07-11,0.0,0.995,0.01,0.00,False,False
4,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.5,2.5,1d,2026-07-13,2026-07-13,0.0,0.995,0.01,0.00,False,False
5,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.5,2.5,final,2026-07-14,2026-07-13,1.0,0.995,0.01,0.00,False,False
6,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.6,2.6,21d,2026-06-23,2026-06-23,0.0,0.905,0.17,0.00,False,False
7,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.6,2.6,14d,2026-06-30,2026-06-25,5.0,0.995,0.01,90.00,False,False
8,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.6,2.6,7d,2026-07-07,2026-07-02,5.0,0.995,0.01,0.00,False,False
9,KXCPIYOY-26JUN,KXCPIYOY-26JUN-T2.6,2.6,3d,2026-07-11,2026-07-11,0.0,0.995,0.01,0.00,False,False


---
# Forecast construction, Step 4 - Forecast panel (survival curve, monotonicity, differencing, summary stats)
---

# Kalshi CPI YoY - Forecast Panel (survival curve, monotonicity, differencing, summary stats)

**Pipeline, per (event, horizon):**
1. Build the raw survival curve S(t) = P(CPI > t) directly from mid-quote snapshots
2. Fix monotonicity violations with isotonic regression (S must weakly decrease as t rises)
3. Difference adjacent thresholds into outcome-bin probabilities, with explicit open tails
4. Compute median, mode, tail probabilities, and an implied mean under a documented tail assumption

**Outputs:** `forecast_panel_raw.csv`, `forecast_panel_adjusted.csv`, `forecast_panel_summary.csv`

**Working defaults used**
- Rows with `is_missing=True` are excluded from the curve entirely (not interpolated)
- `is_stale` rows are KEPT but not treated specially yet -- worth revisiting whether to
  exclude them once you see how much they affect specific events
- Monotonicity fix: isotonic regression (`sklearn.isotonic.IsotonicRegression`,
  `increasing=False`), clipped to [0, 1]
- Open tail assumption for the implied mean: each tail is assigned a probability mass
  (`1 - S(first)` for the lower tail, `S(last)` for the upper tail) and treated as if
  centered half a bin-width beyond the outermost observed threshold. This is a simplifying
  assumption, not a modeled tail distribution -- flag this explicitly if asked, per the
  'open-ended tails' risk already named in the Week 1 proposal.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.isotonic import IsotonicRegression
import os

OUTPUT_DIR = "kalshi_data"
SNAPSHOT_CSV = os.path.join(OUTPUT_DIR, "fixed_horizon_snapshots.csv")

snap = pd.read_csv(SNAPSHOT_CSV)
usable = snap[~snap["is_missing"]].copy()
print(f"Loaded {len(snap)} snapshot rows, {len(usable)} usable (non-missing)")

Loaded 3618 snapshot rows, 3572 usable (non-missing)


## Build the pipeline: survival curve -> monotonicity fix -> differencing -> summary stats, per (event, horizon)

In [ ]:
raw_rows = []
adjusted_rows = []
summary_rows = []

for (event, horizon), group in usable.groupby(["event_ticker", "horizon"]):
    group = group.sort_values("threshold")
    thresholds = group["threshold"].values
    s_raw = group["mid_quote"].values  # S(t) = P(X > t), should be non-increasing

    # ---- 1. RAW survival curve ----
    for t, s in zip(thresholds, s_raw):
        raw_rows.append({"event_ticker": event, "horizon": horizon, "threshold": t, "S_raw": s})

    # ---- 2. Enforce monotonicity via isotonic regression ----
    if len(thresholds) >= 2:
        iso = IsotonicRegression(increasing=False, y_min=0, y_max=1)
        s_adj = iso.fit_transform(thresholds, s_raw)
    else:
        s_adj = s_raw

    n_violations = int(np.sum(np.diff(s_raw) > 1e-9))

    for t, s in zip(thresholds, s_adj):
        adjusted_rows.append({"event_ticker": event, "horizon": horizon, "threshold": t, "S_adjusted": s})

    # ---- 3. Difference into outcome-bin probabilities ----
    bin_probs = -np.diff(s_adj)       # P(t_i < X <= t_i+1)
    lower_tail = 1 - s_adj[0]         # P(X <= first threshold)
    upper_tail = s_adj[-1]            # P(X > last threshold)

    # ---- 4. Summary stats ----
    median = None
    for i in range(len(thresholds) - 1):
        if s_adj[i] >= 0.5 >= s_adj[i+1]:
            if s_adj[i] == s_adj[i+1]:
                median = thresholds[i]
            else:
                frac = (s_adj[i] - 0.5) / (s_adj[i] - s_adj[i+1])
                median = thresholds[i] + frac * (thresholds[i+1] - thresholds[i])
            break

    if len(bin_probs) > 0:
        mode_idx = int(np.argmax(bin_probs))
        mode_range = (thresholds[mode_idx], thresholds[mode_idx+1])
    else:
        mode_range = None

    bin_width = np.median(np.diff(thresholds)) if len(thresholds) > 1 else 1.0
    bin_mids = (thresholds[:-1] + thresholds[1:]) / 2
    mean_val = None
    if len(bin_mids) > 0:
        lower_tail_mid = thresholds[0] - bin_width / 2
        upper_tail_mid = thresholds[-1] + bin_width / 2
        all_mids = np.concatenate([[lower_tail_mid], bin_mids, [upper_tail_mid]])
        all_probs = np.concatenate([[lower_tail], bin_probs, [upper_tail]])
        mean_val = float(np.sum(all_mids * all_probs))

    summary_rows.append({
        "event_ticker": event, "horizon": horizon,
        "median": median,
        "mode_low": mode_range[0] if mode_range else None,
        "mode_high": mode_range[1] if mode_range else None,
        "lower_tail_prob": lower_tail, "upper_tail_prob": upper_tail,
        "implied_mean_TAIL_ASSUMED": mean_val,
        "n_thresholds": len(thresholds), "n_monotonicity_violations": n_violations,
    })

raw_df = pd.DataFrame(raw_rows)
adj_df = pd.DataFrame(adjusted_rows)
summary_df = pd.DataFrame(summary_rows)

print(f"raw: {raw_df.shape}, adjusted: {adj_df.shape}, summary: {summary_df.shape}")

raw: (3572, 4), adjusted: (3572, 4), summary: (261, 10)


In [ ]:
raw_df.to_csv(os.path.join(OUTPUT_DIR, "forecast_panel_raw.csv"), index=False)
adj_df.to_csv(os.path.join(OUTPUT_DIR, "forecast_panel_adjusted.csv"), index=False)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "forecast_panel_summary.csv"), index=False)
print("Saved all three files to", OUTPUT_DIR)

Saved all three files to kalshi_data


## checks

In [ ]:
print("Total monotonicity violations found:", summary_df["n_monotonicity_violations"].sum())
print("Groups with at least one violation:", (summary_df["n_monotonicity_violations"] > 0).sum(),
      "of", len(summary_df))
print()
print("Summary stats sample:")
summary_df.head(10)

Total monotonicity violations found: 318
Groups with at least one violation: 149 of 261

Summary stats sample:


,event_ticker,horizon,median,mode_low,mode_high,lower_tail_prob,upper_tail_prob,implied_mean_TAIL_ASSUMED,n_thresholds,n_monotonicity_violations
0,CPIYOY-22DEC,14d,6.200000,6.1,6.2,0.035,0.203,6.4700,15,2
1,CPIYOY-22DEC,1d,6.343636,6.3,6.4,0.025,0.005,6.3530,15,0
2,CPIYOY-22DEC,21d,6.354545,6.3,6.4,0.065,0.005,6.3510,15,0
3,CPIYOY-22DEC,3d,6.358182,6.3,6.4,0.015,0.005,6.3620,15,0
4,CPIYOY-22DEC,7d,6.362500,6.3,6.4,0.030,0.009,6.3730,15,1
5,CPIYOY-22DEC,final,6.343636,6.3,6.4,0.025,0.005,6.3530,15,0
6,CPIYOY-22NOV,1d,7.148000,7.2,7.3,0.040,0.020,7.1540,13,0
7,CPIYOY-22NOV,3d,7.148000,7.2,7.3,0.045,0.015,7.1545,13,0
8,CPIYOY-22NOV,final,7.148000,7.2,7.3,0.040,0.020,7.1540,13,0
9,CPIYOY-23APR,14d,4.934043,4.9,5.0,0.005,0.030,4.8955,22,3


## One manually-verified release, traced by hand

The plan doc asks for one release traced end-to-end: raw threshold prices -> adjusted survival curve -> final probability distribution. Using the most recent, most liquid event (`KXCPIYOY-26JUN`, `final` horizon) since it's the easiest to eyeball and cross-check against the live Kalshi site directly.

In [ ]:
trace_event, trace_horizon = "KXCPIYOY-26JUN", "final"

print("=== RAW threshold prices (mid-quote) ===")
print(raw_df[(raw_df.event_ticker==trace_event) & (raw_df.horizon==trace_horizon)]
      .sort_values("threshold").to_string(index=False))

print("\n=== ADJUSTED survival curve (after isotonic regression) ===")
print(adj_df[(adj_df.event_ticker==trace_event) & (adj_df.horizon==trace_horizon)]
      .sort_values("threshold").to_string(index=False))

print("\n=== Final summary stats ===")
print(summary_df[(summary_df.event_ticker==trace_event) & (summary_df.horizon==trace_horizon)]
      .to_string(index=False))

=== RAW threshold prices (mid-quote) ===
  event_ticker horizon  threshold  S_raw
KXCPIYOY-26JUN   final        2.5  0.995
KXCPIYOY-26JUN   final        2.6  0.995
KXCPIYOY-26JUN   final        2.7  0.995
KXCPIYOY-26JUN   final        2.8  0.995
KXCPIYOY-26JUN   final        2.9  0.995
KXCPIYOY-26JUN   final        3.0  0.995
KXCPIYOY-26JUN   final        3.1  0.995
KXCPIYOY-26JUN   final        3.2  0.995
KXCPIYOY-26JUN   final        3.3  0.995
KXCPIYOY-26JUN   final        3.4  0.995
KXCPIYOY-26JUN   final        3.5  0.995
KXCPIYOY-26JUN   final        3.6  0.965
KXCPIYOY-26JUN   final        3.7  0.695
KXCPIYOY-26JUN   final        3.8  0.250
KXCPIYOY-26JUN   final        3.9  0.035
KXCPIYOY-26JUN   final        4.0  0.010
KXCPIYOY-26JUN   final        4.1  0.005
KXCPIYOY-26JUN   final        4.2  0.005
KXCPIYOY-26JUN   final        4.3  0.005
KXCPIYOY-26JUN   final        4.4  0.005
KXCPIYOY-26JUN   final        4.5  0.005

=== ADJUSTED survival curve (after isotonic regression) 